# Forge Workshop 01 - Approach B: Just Ask an LLM
### Zero-shot, few-shot and balanced few-shot on MELD (Davit, 6:38 to 7:03 PM)

No training. We write a prompt, send it to Gemini, and read back an emotion for each line.

**How to use:** run the cells from top to bottom. Everything below is instant except Step 8, which is the one live call.

- Every run scores the **same fixed 100 test lines** (the first 100 of MELD's official test set).
- Lines are sent **20 per request**, so 100 lines is 5 requests.
- Steps 4 to 6 read the committed results by default, so nobody waits. Set `USE_SAVED = False` in Step 1 to call Gemini yourself.
- You need your own free key from [Google AI Studio](https://aistudio.google.com/apikey) only for Step 8.

## Step 0 - Setup
If you already ran the base notebook in this session, this cell does nothing. Otherwise it repeats the base notebook setup: get the repo, load MELD, use the official split, and define `save_predictions()`.

In [ ]:
import os, sys, subprocess

if "X_test" in globals() and "save_predictions" in globals():
    print("Base notebook already loaded - skipping setup")
else:
    REPO_URL = "https://github.com/sagarh12/forge-workshop.git"
    REPO_DIR = "forge-workshop"
    if "google.colab" in sys.modules and not os.path.exists("data/processed"):
        if not os.path.exists(REPO_DIR):
            subprocess.check_call(["git", "clone", "-q", REPO_URL])
        os.chdir(REPO_DIR)
    for folder in ["outputs/predictions", "outputs/figures", "notebooks/davit"]:
        os.makedirs(folder, exist_ok=True)

    import pandas as pd

    EXPECTED_LABELS = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}
    meld = pd.read_csv("data/processed/meld_clean.csv")

    # MELD's official split. Never re-split.
    train_df = meld[meld["split"] == "train"].reset_index(drop=True)
    test_df  = meld[meld["split"] == "test"].reset_index(drop=True)
    X_train, y_train = train_df["utterance"].tolist(), train_df["label"].tolist()
    X_test,  y_test  = test_df["utterance"].tolist(),  test_df["label"].tolist()

    ALLOWED_MODELS = {"bilstm", "bert_finetuned", "zero_shot", "few_shot", "balanced_few_shot"}

    def save_predictions(X, y_true, y_pred, model_name: str, full_test_set: bool = True):
        """Same function as the base notebook (Step 8)."""
        assert model_name in ALLOWED_MODELS, f"model_name must be one of {sorted(ALLOWED_MODELS)}"
        assert len(X) == len(y_true) == len(y_pred), "X, y_true and y_pred must be the same length"
        n = len(X)
        assert list(X) == X_test[:n], "Utterances do not match the official test set order"
        assert list(y_true) == y_test[:n], "true_label does not match the official test labels"
        if full_test_set:
            assert n == len(X_test), f"Expected {len(X_test)} rows (full test set), got {n}"
        bad = set(y_pred) - EXPECTED_LABELS
        assert not bad, f"Predicted labels outside the seven emotions: {bad}"
        df = pd.DataFrame({"utterance": list(X), "true_label": list(y_true),
                           "predicted_label": list(y_pred), "model_name": model_name})
        path = f"outputs/predictions/{model_name}_predictions.csv"
        df.to_csv(path, index=False)
        print(f"Predictions saved to {path} ({n} rows)")
        return df

    print(f"Setup done. Train lines: {len(X_train)}  Test lines: {len(X_test)}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai", "imbalanced-learn"], check=False)

import json, time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

## Step 1 - Settings and the fixed 100 test lines
Everyone scores the **same 100 lines**: the first 100 of the official test set, in the official order. Sheheryar's scoreboard uses these same lines.

`USE_SAVED = True` makes Steps 4 to 6 read the committed results instead of calling Gemini. Leave it on in class so the block is instant. Turn it off if you want to run all three yourself.

In [ ]:
USE_SAVED  = True              # True = read committed CSVs (instant). False = call Gemini for Steps 4-6.
MODEL      = "gemini-3.5-flash"  # tested on a free key, 25 Nov 2026
N_DEMO     = 100               # the fixed 100 lines everyone is scored on
N_LIVE     = 40                # Step 8 only: smaller so the live demo finishes in about 30 seconds
BATCH_SIZE = 20                # lines per request
LABELS     = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

X_demo, y_demo = X_test[:N_DEMO], y_test[:N_DEMO]

print("Emotion mix in the 100 lines:")
print(pd.Series(y_demo).value_counts().to_string())
print()
for text, label in list(zip(X_demo, y_demo))[:5]:
    print(f"[{label:8}] {text}")

# The naive baseline: always answer the most common emotion
majority = pd.Series(y_train).value_counts().idxmax()
baseline_acc = accuracy_score(y_demo, [majority] * N_DEMO)
print(f"\nNaive baseline (always say '{majority}'): {baseline_acc:.0%} accuracy")

## Step 2 - Your own Gemini key
Create a free key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey) and paste it in the box below. The box hides what you type. **Never paste a key into a cell.**

Use a **personal Google account**. School and work accounts are usually blocked from creating the project a key needs.

Only Step 8 needs this. If you are just reading the saved results, you can skip it.

In [ ]:
from getpass import getpass
from google import genai

# Colab can remember the key for you: key icon in the left sidebar -> add GEMINI_API_KEY -> Notebook access on.
try:
    from google.colab import userdata
    os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
except Exception:
    pass

client = None
if not os.environ.get("GEMINI_API_KEY"):
    try:
        pasted = getpass("Paste your Gemini API key (or press Enter to skip): ").strip()
    except Exception:
        pasted = ""
    if pasted:
        os.environ["GEMINI_API_KEY"] = pasted

if os.environ.get("GEMINI_API_KEY"):
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    print("Key loaded. Using model:", MODEL)
else:
    print("No key given. Saved results will be used; Step 8 will not run.")

## Step 3 - The batched runner
One request carries 20 lines and Gemini answers with a JSON list of 20 emotions.

Three safety nets, because a free key on a room full of laptops will fail for somebody:
1. If `USE_SAVED` is on, no call is made at all.
2. If a call is rate limited or the server is busy, it waits and tries again.
3. If it still fails, the committed results are loaded and the block carries on.

In [ ]:
PAUSE_SECONDS = 4    # wait between requests
RETRIES       = 4    # attempts per batch
SLOW_ERRORS   = ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL")

SYNONYMS = {"angry": "anger", "happy": "joy", "happiness": "joy", "sad": "sadness",
            "surprised": "surprise", "disgusted": "disgust", "scared": "fear", "afraid": "fear"}

def clean_label(x):
    x = str(x).strip().lower()
    x = SYNONYMS.get(x, x)
    return x if x in LABELS else "neutral"

def call_gemini(prompt):
    """The only place we talk to the API."""
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"temperature": 0, "response_mime_type": "application/json"},
    )
    return response.text

def classify_batch(instructions, lines):
    numbered = "\n".join(f"{i + 1}. {text}" for i, text in enumerate(lines))
    prompt = (
        f"{instructions}\n\n"
        f"Classify each of the {len(lines)} lines below.\n"
        f"Reply with only a JSON list of exactly {len(lines)} strings, one emotion per line, in the same order.\n\n"
        f"{numbered}"
    )
    last_error = None
    for attempt in range(RETRIES):
        try:
            answer = json.loads(call_gemini(prompt))
            if isinstance(answer, dict):                 # e.g. {"labels": [...]}
                answer = next(v for v in answer.values() if isinstance(v, list))
            if len(answer) != len(lines):
                raise ValueError(f"expected {len(lines)} labels, got {len(answer)}")
            return [clean_label(a) for a in answer]
        except Exception as e:
            last_error = e
            if attempt < RETRIES - 1:
                wait = 20 if any(s in str(e) for s in SLOW_ERRORS) else 3
                print(f"   busy, waiting {wait}s ({str(e)[:70]})")
                time.sleep(wait)
    raise last_error

def score(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)
    print(f"{name}: accuracy {acc:.0%} | macro F1 {f1:.2f}")
    return acc, f1

def load_saved(name, n):
    path = f"outputs/predictions/{name}_predictions.csv"
    if not os.path.exists(path):
        return None
    saved = pd.read_csv(path).head(n)
    if saved["utterance"].tolist() != X_test[:n]:
        print(f"   {path} does not match the fixed lines - ignoring it")
        return None
    return saved["predicted_label"].tolist()

results = {}

def run_style(instructions, name, n=None, save=True, force_live=False):
    """Score one prompt style. Reads the committed CSV unless asked to go live."""
    n = n or N_DEMO
    X, y_true = X_test[:n], y_test[:n]

    if USE_SAVED and not force_live:
        y_pred = load_saved(name, n)
        if y_pred:
            print(f"Loaded committed results for {name} ({n} lines, no API call)")
            results[name] = score(y_true, y_pred, name)
            return y_pred
        print(f"No committed results for {name} - calling Gemini instead")

    if client is None:
        print("No Gemini key, so this cell is skipped. Run Step 2 and paste a key if you want to try it live.")
        return []

    start, y_pred = time.time(), []
    try:
        for i in range(0, n, BATCH_SIZE):
            y_pred += classify_batch(instructions, X[i:i + BATCH_SIZE])
            print(f"   {len(y_pred)}/{n} lines done")
            if len(y_pred) < n:
                time.sleep(PAUSE_SECONDS)
        print(f"   finished in {time.time() - start:.0f}s")
        if save:
            save_predictions(X, y_true, y_pred, name, full_test_set=False)
    except Exception as e:
        print(f"   Live call failed: {str(e)[:110]}")
        y_pred = load_saved(name, n)
        if not y_pred:
            print("   Nothing committed for this one. Wait a minute and run the cell again.")
            return []
        print(f"   -> Loaded the committed results instead")
    results[name] = score(y_true, y_pred, name)
    return y_pred

## Step 4 - Zero-shot
**Zero-shot** means we give the model the task and the seven emotion names, and **no examples at all**.

In [ ]:
ZERO_SHOT_PROMPT = """You are reading lines of dialogue from a TV show.
Decide which emotion the speaker is expressing in each line.
Choose exactly one of: anger, disgust, fear, joy, neutral, sadness, surprise."""

print(ZERO_SHOT_PROMPT)
print("-" * 60)
pred_zero = run_style(ZERO_SHOT_PROMPT, "zero_shot")

## Step 5 - Few-shot
**Few-shot** means we show the model some example lines with the correct emotion before asking.

The examples come from the **training lines only**, never from the test lines. Here we pick 21 at random, so they follow the natural mix of the data, which is mostly `neutral`.

In [ ]:
N_EXAMPLES = 21   # same number in both few-shot styles, so the only difference is the balance

# Training lines only; nothing that appears in the 100 test lines; readable lengths
pool = train_df[~train_df["utterance"].isin(set(X_demo))]
pool = pool[pool["utterance"].str.len().between(15, 120)].drop_duplicates("utterance")

def format_examples(df):
    return "\n".join(f'Line: {u}\nEmotion: {l}\n' for u, l in zip(df["utterance"], df["label"]))

random_examples = pool.sample(N_EXAMPLES, random_state=42)
print("Emotions in the random examples:")
print(random_examples["label"].value_counts().to_string())

FEW_SHOT_PROMPT = ZERO_SHOT_PROMPT + "\n\nHere are some examples:\n\n" + format_examples(random_examples)
print("-" * 60)
print(FEW_SHOT_PROMPT)
print("-" * 60)
pred_few = run_style(FEW_SHOT_PROMPT, "few_shot")

## Step 6 - Balanced few-shot
Same idea, same number of examples, but now **3 examples for each of the seven emotions**. The rare ones, `fear` and `disgust`, are guaranteed to appear.

In [ ]:
balanced_examples = pool.groupby("label", group_keys=False).sample(N_EXAMPLES // len(LABELS), random_state=42)
balanced_examples = balanced_examples.sample(frac=1, random_state=42)   # shuffle so the order gives nothing away
print("Emotions in the balanced examples:")
print(balanced_examples["label"].value_counts().to_string())

BALANCED_PROMPT = ZERO_SHOT_PROMPT + "\n\nHere are some examples:\n\n" + format_examples(balanced_examples)
print("-" * 60)
print(BALANCED_PROMPT)
print("-" * 60)
pred_balanced = run_style(BALANCED_PROMPT, "balanced_few_shot")

## Step 7 - Compare the three
Same 100 lines, three prompts. The dashed line is the naive baseline that always answers `neutral`.

Watch the two bars for each prompt. **Accuracy** barely moves, but **macro F1**, which counts the rare emotions as much as the common ones, moves a lot more. That is Sheheryar's point arriving early: on imbalanced data, accuracy hides things.

In [ ]:
table = pd.DataFrame(results, index=["accuracy", "macro_f1"]).T
print(table.round(2).to_string())

x = range(len(table))
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar([i - 0.2 for i in x], table["accuracy"], 0.4, label="Accuracy",  color="#1424D0")
b2 = ax.bar([i + 0.2 for i in x], table["macro_f1"], 0.4, label="Macro F1", color="#8E9BFF")
ax.axhline(baseline_acc, color="gray", linestyle="--", label=f"always '{majority}' ({baseline_acc:.0%})")
for bars in (b1, b2):
    for b in bars:
        ax.annotate(f"{b.get_height():.2f}", (b.get_x() + b.get_width() / 2, b.get_height()),
                    ha="center", va="bottom", fontsize=10)
ax.set_xticks(list(x)); ax.set_xticklabels(table.index)
ax.set_ylim(0, 1); ax.set_ylabel("Score on the 100 test lines")
ax.set_title("No training at all: three prompts vs the naive baseline", fontweight="bold")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("outputs/figures/llm_prompt_comparison.png", dpi=150)
plt.show()

# Which emotions does each prompt actually find? (recall per emotion)
recall = pd.DataFrame({
    name: [sum(p == t == lab for p, t in zip(pred, y_demo)) / max(1, y_demo.count(lab)) for lab in LABELS]
    for name, pred in [("zero_shot", pred_zero), ("few_shot", pred_few), ("balanced_few_shot", pred_balanced)]
}, index=LABELS)
recall["lines in test"] = [y_demo.count(lab) for lab in LABELS]
print("\nShare of each emotion found correctly:")
print(recall.round(2).to_string())

## Step 8 - Live prompt surgery (your turn)
This is the one cell that calls Gemini live, on **40 lines** so it finishes in about half a minute.

Change **one line** of the prompt below and run the cell. Did the score go up or down?

Ideas: explain what `neutral` means, say these are jokes from a sitcom, or warn it about sarcasm.

In [ ]:
MY_PROMPT = """You are reading lines of dialogue from a TV show.
Decide which emotion the speaker is expressing in each line.
Choose exactly one of: anger, disgust, fear, joy, neutral, sadness, surprise.
Only answer neutral when the line shows no clear emotion at all."""   # <- the line we added. Change it!

pred_mine = run_style(MY_PROMPT, "my_prompt", n=N_LIVE, save=False, force_live=True)

if pred_mine:
    score(y_test[:N_LIVE], pred_zero[:N_LIVE], "zero-shot on the same 40 lines, for comparison")

## Step 9 - Balancing the data
MELD is badly imbalanced: `neutral` dominates and `fear` and `disgust` are rare. A model trained on this learns to answer `neutral` too often.

Two simple fixes for the **training set**: **oversampling** repeats rare emotions until every class matches the biggest, and **undersampling** drops common ones until every class matches the smallest. We only show the idea. Nothing is retrained.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

idx = pd.DataFrame({"row": range(len(y_train))})
_, y_over  = RandomOverSampler(random_state=42).fit_resample(idx, y_train)
_, y_under = RandomUnderSampler(random_state=42).fit_resample(idx, y_train)

counts = pd.DataFrame({
    "Before": pd.Series(y_train).value_counts(),
    "Oversampled": pd.Series(y_over).value_counts(),
    "Undersampled": pd.Series(y_under).value_counts(),
}).loc[pd.Series(y_train).value_counts().index]
print(counts.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=True)
for ax, col in zip(axes, counts.columns):
    bars = ax.bar(counts.index, counts[col], color="#1424D0")
    for b in bars:
        ax.annotate(f"{int(b.get_height()):,}", (b.get_x() + b.get_width() / 2, b.get_height()),
                    ha="center", va="bottom", fontsize=8)
    ax.set_title(f"{col}\n{counts[col].sum():,} training lines", fontweight="bold")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Training lines")
plt.tight_layout()
plt.savefig("outputs/figures/balancing_before_after.png", dpi=150)
plt.show()
print("\nOversampling invents no new lines, it repeats the rare ones.")
print("Undersampling throws away most of the data: 9,989 lines become 1,876.")

## Optional - Does conversation context help?
So far the model saw one line at a time. Here it also sees up to three earlier lines of the same conversation, on 40 lines. Skip this if time is short. Nothing is saved.

In [ ]:
ctx = pd.read_csv("data/processed/meld_context.csv")
ctx = ctx[ctx["split"] == "test"].reset_index(drop=True).head(N_LIVE)
assert ctx["utterance"].tolist() == X_test[:N_LIVE], "Context file is not in the official test order"

with_context = [f"(earlier: {c}) NOW CLASSIFY: {u}" if isinstance(c, str) and c else u
                for c, u in zip(ctx["context"], ctx["utterance"])]

CONTEXT_PROMPT = ZERO_SHOT_PROMPT + "\nSome lines include earlier lines of the conversation in brackets. Classify only the line after NOW CLASSIFY."
try:
    y_ctx = []
    for i in range(0, N_LIVE, BATCH_SIZE):
        y_ctx += classify_batch(CONTEXT_PROMPT, with_context[i:i + BATCH_SIZE])
        print(f"   {len(y_ctx)}/{N_LIVE} lines done")
    score(y_test[:N_LIVE], y_ctx, "with context")
    score(y_test[:N_LIVE], pred_zero[:N_LIVE], "without context")
except Exception as e:
    print("Context run skipped:", str(e)[:110])